# RSWQ Tabular Dataset Assembly

Notebook นี้ใช้รวม CSV จาก 2 แหล่งหลัก:

1. `Sentinel-2 + Ground Observation` จาก notebook satellite feature extraction
2. `Context features` จาก GSMaP + ERA5-Land Hourly + static geospatial context

ผลลัพธ์คือ CSV กลางสำหรับ train baseline ML / MLP แบบ tabular โดยยังไม่ทำ model-specific preprocessing เช่น imputation, scaling หรือ target encoding


## Workflow

```text
Sentinel-2 model dataset
    + strict/relaxed candidate sample_id
    + context feature CSV
        ↓ merge by sample_id
Full / Strict / Relaxed assembled datasets
        ↓
Feature catalog + summary report
        ↓
Training notebook for ML / MLP
```

หมายเหตุ: ไฟล์ output ยังเก็บ target และ metadata ไว้ครบ เพื่อให้เลือก target ใน training notebook ได้ แต่ต้องใช้ `feature_catalog` เพื่อตัด target/leakage ออกจาก input feature ก่อน train


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)


## 1. Configure Input and Output Paths


In [ ]:
# Default paths for the files currently in E:/Downloads.
# If running in Colab, upload/copy the CSVs and change BASE accordingly.
BASE = Path(r"E:\Downloads")

S2_FULL_CSV = BASE / "sentinel2_model_dataset_epo14_2562_2568_b200_watermask_v2_s2formula.csv"
S2_STRICT_CSV = BASE / "sentinel2_train_strict_candidates_epo14_2562_2568_b200_watermask_v2_s2formula.csv"
S2_RELAXED_CSV = BASE / "sentinel2_train_relaxed_candidates_epo14_2562_2568_b200_watermask_v2_s2formula.csv"
CONTEXT_CSV = BASE / "rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv"

OUT_DIR = BASE / "rswq_tabular_assembly_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_LABEL = "2562_2568"

print("Output directory:", OUT_DIR)
for path in [S2_FULL_CSV, S2_STRICT_CSV, S2_RELAXED_CSV, CONTEXT_CSV]:
    print(path.name, "exists:", path.exists())


## 2. Load CSV Files


In [ ]:
def load_csv(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


s2_full = load_csv(S2_FULL_CSV)
s2_strict = load_csv(S2_STRICT_CSV)
s2_relaxed = load_csv(S2_RELAXED_CSV)
context = load_csv(CONTEXT_CSV)

print("s2_full:", s2_full.shape)
print("s2_strict:", s2_strict.shape)
print("s2_relaxed:", s2_relaxed.shape)
print("context:", context.shape)


## 3. Validate `sample_id` Before Merge


In [ ]:
def assert_unique_sample_id(df, name):
    if "sample_id" not in df.columns:
        raise KeyError(f"{name} has no sample_id column")
    duplicate_count = int(df["sample_id"].duplicated().sum())
    if duplicate_count:
        raise ValueError(f"{name} has duplicated sample_id rows: {duplicate_count}")


for name, df in {
    "s2_full": s2_full,
    "s2_strict": s2_strict,
    "s2_relaxed": s2_relaxed,
    "context": context,
}.items():
    assert_unique_sample_id(df, name)
    print(name, "rows:", len(df), "unique sample_id:", df["sample_id"].nunique())


full_ids = set(s2_full["sample_id"].astype(str))
strict_ids = set(s2_strict["sample_id"].astype(str))
relaxed_ids = set(s2_relaxed["sample_id"].astype(str))
context_ids = set(context["sample_id"].astype(str))

print("Strict subset of full:", strict_ids.issubset(full_ids))
print("Relaxed subset of full:", relaxed_ids.issubset(full_ids))
print("Full and context IDs equal:", full_ids == context_ids)

if not strict_ids.issubset(full_ids):
    raise ValueError("Strict sample_id set is not a subset of s2_full")
if not relaxed_ids.issubset(full_ids):
    raise ValueError("Relaxed sample_id set is not a subset of s2_full")
if full_ids != context_ids:
    raise ValueError("Context sample_id set does not exactly match s2_full")


## 4. Define Targets, Leakage Columns, and Helper Functions


In [ ]:
TARGET_COLS = [
    "turbidity_NTU_clean",
    "SS_mg_l_clean",
    "DO_mg_l_clean",
    "BOD_mg_l_clean",
    "NH3_N_mg_l_clean",
    "TCB_MPN_100ml_clean",
    "FCB_MPN_100ml_clean",
    "WQI_clean",
]

FIELD_AUX_TARGET_CANDIDATES = [
    "pH_clean",
    "conductivity_clean",
    "salinity_ppt_clean",
    "temp_air_clean",
    "temp_water_clean",
]

WQI_LEAKAGE_COLS = [
    "WQI_DO_score_pcd5",
    "WQI_BOD_score_pcd5",
    "WQI_TCB_score_pcd5",
    "WQI_FCB_score_pcd5",
    "WQI_NH3_score_pcd5",
    "WQI_reported",
    "WQI_score_input_complete",
    "WQI_mean_subindex_pcd5",
    "WQI_mean_level_pcd5",
    "WQI_worst_parameter_level_pcd5",
    "WQI_adjustment_pcd5",
    "WQI_recalc_pcd5",
    "WQI_source",
    "WQI_recalc_status",
]

SKEWED_TARGETS_FOR_LOG1P = [
    "turbidity_NTU_clean",
    "SS_mg_l_clean",
    "BOD_mg_l_clean",
    "NH3_N_mg_l_clean",
    "TCB_MPN_100ml_clean",
    "FCB_MPN_100ml_clean",
]


def stable_fold(value, n_folds=5):
    text = "" if pd.isna(value) else str(value)
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % n_folds


## 5. Merge Sentinel-2 and Context Features


In [ ]:
def add_subset_flags(df, strict_ids, relaxed_ids):
    out = df.copy()
    out["is_s2_strict_candidate"] = out["sample_id"].astype(str).isin(strict_ids)
    out["is_s2_relaxed_candidate"] = out["sample_id"].astype(str).isin(relaxed_ids)
    out["s2_candidate_set"] = np.select(
        [
            out["is_s2_strict_candidate"],
            out["is_s2_relaxed_candidate"],
        ],
        [
            "strict",
            "relaxed_only",
        ],
        default="full_only",
    )
    return out


def merge_context(s2_df, context_df):
    # Context CSV has duplicate metadata already present in the Sentinel-2 table.
    # Keep Sentinel-2/Ground Observation metadata as the authoritative copy,
    # and add only context-specific columns from context_df.
    duplicate_context_cols = sorted((set(s2_df.columns) & set(context_df.columns)) - {"sample_id"})
    context_add = context_df.drop(columns=duplicate_context_cols)
    print("Dropped duplicate context metadata columns:", duplicate_context_cols)
    print("Context feature columns added:", len(context_add.columns) - 1)

    out = s2_df.merge(context_add, on="sample_id", how="left", validate="one_to_one")
    out["has_context_match"] = out["sample_id"].isin(context_df["sample_id"])
    return out


s2_full_flagged = add_subset_flags(s2_full, strict_ids, relaxed_ids)
assembled_full = merge_context(s2_full_flagged, context)

print("assembled_full:", assembled_full.shape)
print("context matched:", assembled_full["has_context_match"].value_counts(dropna=False).to_dict())


## 6. Add QA Helpers, Split Labels, and Target Transforms


In [ ]:
def postprocess_assembled(df):
    out = df.copy()

    # Clip tiny negative ERA5-Land runoff/solar numerical artifacts.
    clip_cols = [
        col for col in out.columns
        if col.startswith("era5h_runoff_") or col.startswith("era5h_solar_")
    ]
    for col in clip_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").clip(lower=0)

    # Missingness flags for JRC static water-history features.
    for col in [
        "jrc_occurrence_mean_500m",
        "jrc_seasonality_mean_500m",
        "jrc_recurrence_mean_500m",
    ]:
        if col in out.columns:
            out[f"{col}_missing"] = out[col].isna().astype("int8")

    # Temporal split labels.
    if "year_be" in out.columns:
        year = pd.to_numeric(out["year_be"], errors="coerce")
        out["split_temporal_train_2562_2566_test_2567_2568"] = np.where(
            year <= 2566,
            "train",
            np.where(year >= 2567, "test", "unknown"),
        )
        out["split_temporal_train_2562_2565_val_2566_test_2567_2568"] = np.select(
            [year <= 2565, year.eq(2566), year >= 2567],
            ["train", "val", "test"],
            default="unknown",
        )

    # Deterministic CV folds.
    out["cv_fold_random5_sample_id"] = out["sample_id"].map(stable_fold).astype("int8")
    if "station_canonical" in out.columns:
        station_fold = {
            station: stable_fold(station)
            for station in sorted(out["station_canonical"].dropna().astype(str).unique())
        }
        out["cv_fold_station5"] = out["station_canonical"].astype(str).map(station_fold).astype("int8")

    # Target transforms for skewed targets. These are labels, not input features.
    for col in SKEWED_TARGETS_FOR_LOG1P:
        if col in out.columns:
            numeric = pd.to_numeric(out[col], errors="coerce")
            out[f"log1p_{col}"] = np.where(numeric >= 0, np.log1p(numeric), np.nan)

    return out


assembled_full = postprocess_assembled(assembled_full)
assembled_strict = assembled_full[assembled_full["sample_id"].astype(str).isin(strict_ids)].copy().reset_index(drop=True)
assembled_relaxed = assembled_full[assembled_full["sample_id"].astype(str).isin(relaxed_ids)].copy().reset_index(drop=True)

print("full:", assembled_full.shape)
print("strict:", assembled_strict.shape)
print("relaxed:", assembled_relaxed.shape)


## 7. Create Feature Catalog


In [ ]:
def classify_col(col):
    if col == "sample_id":
        return "identifier", False, "join key only"

    if col in TARGET_COLS:
        return "target", False, "ground-truth target; never use as feature for that target"

    if col in FIELD_AUX_TARGET_CANDIDATES:
        return "field_measurement_aux_or_target", False, "observed field measurement; use carefully"

    if col in WQI_LEAKAGE_COLS:
        return "wqi_leakage_or_aux", False, "leaks WQI if predicting WQI"

    if col.startswith("log1p_"):
        return "target_transform", False, "transformed target candidate, not an input feature"

    metadata_cols = {
        "station_survey_id", "canonical_station_survey_id", "station_original",
        "station_canonical", "survey_no", "field_date", "sat_date",
        "field_date_dt", "sat_date_dt", "image_id", "source_chunk_file",
        "water_body", "coordinate_waterbody", "coordinate_assignment_status",
        "station_location", "survey_time", "survey_time_clean",
        "sample_datetime_local", "sample_datetime_utc",
        "sample_cutoff_local", "sample_cutoff_utc", "season_south_th",
    }
    if col in metadata_cols:
        return "metadata", False, "metadata or categorical identifier; encode deliberately if used"

    qa_cols = {
        "match_type", "selection_reason", "match_window", "data_quality",
        "has_image_match", "has_band_value", "has_clear_pixel", "has_water_pixel",
        "usable_s2", "usable_s2_water", "is_s2_strict_candidate",
        "is_s2_relaxed_candidate", "s2_candidate_set", "has_context_match",
        "candidate_has_enough_water", "exact_date_match",
    }
    if col in qa_cols or col.startswith("split_") or col.startswith("cv_fold_"):
        return "qa_or_split", False, "QA/split column; use for filtering/evaluation"

    qa_numeric = {
        "days_diff", "days_diff_signed", "abs_date_diff_days", "scene_cloud_pct",
        "valid_pixel_count", "valid_water_pixel_count", "total_pixel_count",
        "water_pixel_fraction", "water_pixel_pct", "local_cloud_fraction",
        "local_cloud_pct", "candidate_valid_count", "candidate_water_count",
        "local_match_rank", "min_candidate_water_pixels", "buffer_m", "scale_m",
        "primary_window_days", "fallback_window_days", "minutes_after_cutoff",
    }
    if col in qa_numeric:
        return "qa_numeric", False, "image/time QA; useful for filtering or sensitivity checks"

    if col.endswith("_median") and (
        col.startswith("B")
        or col.startswith(("ND", "AWEI", "Red_", "NIR_", "ph_formula", "turbidity_formula",
                           "salinity_formula", "do_formula", "chla_formula", "secchi_formula",
                           "tsi_formula"))
    ):
        return "sentinel2_feature", True, "Sentinel-2 water-masked median feature"

    if col.startswith(("rain_gsmap_", "rainy_hours_gsmap_", "hours_since_last_rain_gsmap_")):
        return "context_rainfall_feature", True, "strict prior rainfall feature"

    if col.startswith("era5h_"):
        return "context_weather_feature", True, "strict prior ERA5-Land hourly feature"

    if col.startswith(("jrc_", "srtm_", "wc_")):
        return "context_static_feature", True, "static geospatial context feature"

    if col in {"month", "day_of_year", "doy_sin", "doy_cos", "lat", "lon", "latitude", "longitude"}:
        return "time_location_feature", True, "time/location feature; validate with station-wise split"

    if col in {"year_be", "year_ad"}:
        return "time_metadata", False, "use for temporal split; avoid as feature unless intentional"

    return "other", False, "review before modeling"


feature_catalog_rows = []
for i, col in enumerate(assembled_full.columns):
    role, recommended, note = classify_col(col)
    feature_catalog_rows.append({
        "column_order": i,
        "column_name": col,
        "role": role,
        "recommended_as_feature_default": recommended,
        "note": note,
    })

feature_catalog = pd.DataFrame(feature_catalog_rows)
display(feature_catalog["role"].value_counts().rename("n_columns"))
display(feature_catalog.head(20))


## 8. Save Assembled CSV Outputs


In [ ]:
output_paths = {
    "full": OUT_DIR / f"rswq_model_input_full_s2_context_{YEAR_LABEL}.csv",
    "strict": OUT_DIR / f"rswq_model_input_strict_s2_context_{YEAR_LABEL}.csv",
    "relaxed": OUT_DIR / f"rswq_model_input_relaxed_s2_context_{YEAR_LABEL}.csv",
    "feature_catalog": OUT_DIR / f"rswq_model_input_feature_catalog_{YEAR_LABEL}.csv",
    "summary": OUT_DIR / f"rswq_model_input_summary_{YEAR_LABEL}.csv",
    "metadata": OUT_DIR / f"rswq_model_input_metadata_{YEAR_LABEL}.json",
}

assembled_full.to_csv(output_paths["full"], index=False, encoding="utf-8-sig")
assembled_strict.to_csv(output_paths["strict"], index=False, encoding="utf-8-sig")
assembled_relaxed.to_csv(output_paths["relaxed"], index=False, encoding="utf-8-sig")
feature_catalog.to_csv(output_paths["feature_catalog"], index=False, encoding="utf-8-sig")

summary_rows = []
for name, df in {
    "full": assembled_full,
    "strict": assembled_strict,
    "relaxed": assembled_relaxed,
}.items():
    summary_rows.append({"dataset": name, "metric": "rows", "value": len(df)})
    summary_rows.append({"dataset": name, "metric": "columns", "value": len(df.columns)})
    summary_rows.append({"dataset": name, "metric": "unique_sample_id", "value": df["sample_id"].nunique()})
    summary_rows.append({"dataset": name, "metric": "stations", "value": df["station_canonical"].nunique()})
    for col in TARGET_COLS + FIELD_AUX_TARGET_CANDIDATES:
        if col in df.columns:
            summary_rows.append({"dataset": name, "metric": f"{col}_nonmissing", "value": int(df[col].notna().sum())})
    if "match_window" in df.columns:
        for label, count in df["match_window"].value_counts(dropna=False).items():
            summary_rows.append({"dataset": name, "metric": f"match_window={label}", "value": int(count)})
    if "data_quality" in df.columns:
        for label, count in df["data_quality"].value_counts(dropna=False).items():
            summary_rows.append({"dataset": name, "metric": f"data_quality={label}", "value": int(count)})

assembly_summary = pd.DataFrame(summary_rows)
assembly_summary.to_csv(output_paths["summary"], index=False, encoding="utf-8-sig")

metadata = {
    "inputs": {
        "s2_full": str(S2_FULL_CSV),
        "s2_strict": str(S2_STRICT_CSV),
        "s2_relaxed": str(S2_RELAXED_CSV),
        "context": str(CONTEXT_CSV),
    },
    "outputs": {name: str(path) for name, path in output_paths.items()},
    "target_columns": TARGET_COLS,
    "field_aux_or_target_candidates": FIELD_AUX_TARGET_CANDIDATES,
    "wqi_leakage_columns": WQI_LEAKAGE_COLS,
    "recommended_feature_count": int(feature_catalog["recommended_as_feature_default"].sum()),
    "notes": [
        "strict is recommended for main modeling",
        "relaxed is recommended for sensitivity analysis",
        "do not use target/WQI leakage columns as model features",
        "imputation/scaling should be done inside the training pipeline",
    ],
}
output_paths["metadata"].write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

for name, path in output_paths.items():
    print(name, path, "size:", path.stat().st_size)


## 9. Quick QA of Final Outputs


In [ ]:
for name, df in {
    "full": assembled_full,
    "strict": assembled_strict,
    "relaxed": assembled_relaxed,
}.items():
    print("\n==", name, "==")
    print("shape:", df.shape)
    print("unique sample_id:", df["sample_id"].nunique())
    print("duplicate sample_id:", int(df["sample_id"].duplicated().sum()))
    print("context match:", df["has_context_match"].value_counts(dropna=False).to_dict())
    if "match_window" in df.columns:
        print("match_window:", df["match_window"].value_counts(dropna=False).to_dict())
    if "data_quality" in df.columns:
        print("data_quality:", df["data_quality"].value_counts(dropna=False).to_dict())
    display(df[TARGET_COLS].notna().sum().rename("target_nonmissing"))


## Final Output Meaning

ใช้ไฟล์เหล่านี้ต่อใน training notebook:

- `rswq_model_input_strict_s2_context_2562_2568.csv`: ชุดหลักสำหรับ modeling คุณภาพสูงกว่า แต่ sample น้อยกว่า
- `rswq_model_input_relaxed_s2_context_2562_2568.csv`: ชุดสำรอง/sensitivity analysis sample มากขึ้น แต่ noise มากขึ้น
- `rswq_model_input_full_s2_context_2562_2568.csv`: ชุดเต็มสำหรับ QA/ทดลอง filter เอง
- `rswq_model_input_feature_catalog_2562_2568.csv`: ใช้เลือก feature และตัด target/leakage

Output ยังไม่ใช่ final tensor สำหรับ MLP เพราะ MLP ต้องทำ imputation/scaling ใน training pipeline เพื่อป้องกัน leakage จากการ fit scaler/imputer บน test set
